# Realistic ML Sweep

This notebook trains a small logistic baseline on synthetic customer-retention data, runs a bounded hyperparameter sweep, repairs a failed sweep cell in place, and checks threshold/error behavior for the selected model.

Artifact scope is intentionally left to Notebook Lens inference: `artifacts/agent_rounds/20260615_round6/ml_sweep_realistic/`.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path
from statistics import mean

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from samples.ml_tasks.common import (  # noqa: E402
    FEATURES,
    classification_metrics,
    display_html,
    generate_classification_rows,
    metrics_table,
    save_json,
    score_rows,
    split_rows,
    train_logistic,
)

ARTIFACT_DIR = Path("artifacts/agent_rounds/20260615_round6/ml_sweep_realistic")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

rows = generate_classification_rows(n=1000, seed=106, drift=0.0)
train_rows, holdout_rows = split_rows(rows, train_frac=0.72, seed=17)
valid_rows, test_rows = split_rows(holdout_rows, train_frac=0.5, seed=23)

baseline_model = train_logistic(train_rows, lr=0.05, epochs=140, l2=0.001)
baseline_valid = classification_metrics(score_rows(valid_rows, baseline_model))
baseline_test = classification_metrics(score_rows(test_rows, baseline_model))

feature_profile = {
    feature: {
        "mean": mean(float(row[feature]) for row in rows),
        "min": min(float(row[feature]) for row in rows),
        "max": max(float(row[feature]) for row in rows),
        "missing": sum(1 for row in rows if row.get(feature) is None),
    }
    for feature in FEATURES
}

dataset_profile = {
    "row_count": len(rows),
    "train_rows": len(train_rows),
    "valid_rows": len(valid_rows),
    "test_rows": len(test_rows),
    "positive_rate": sum(row["label"] for row in rows) / len(rows),
    "features": feature_profile,
}

profile_path = save_json(ARTIFACT_DIR / "dataset_profile.json", dataset_profile)
model_path = save_json(ARTIFACT_DIR / "baseline_model.json", baseline_model)
metrics_path = save_json(
    ARTIFACT_DIR / "baseline_metrics.json",
    {
        "valid": baseline_valid,
        "test": baseline_test,
        "hyperparameters": baseline_model["hyperparameters"],
    },
)

print(f"rows: {len(rows)}")
print(f"train_rows: {len(train_rows)}")
print(f"valid_rows: {len(valid_rows)}")
print(f"test_rows: {len(test_rows)}")
print(f"positive_rate: {dataset_profile['positive_rate']:.4f}")
print(f"profile_artifact: {profile_path}")
print(f"model_artifact: {model_path}")
print(f"metrics_artifact: {metrics_path}")
print("baseline_valid:")
for key, value in baseline_valid.items():
    print(f"- {key}: {value:.4f}" if isinstance(value, float) else f"- {key}: {value}")
print("baseline_test:")
for key, value in baseline_test.items():
    print(f"- {key}: {value:.4f}" if isinstance(value, float) else f"- {key}: {value}")

metrics_table("Baseline Validation Metrics", baseline_valid)
display_html(
    "<h3>Dataset Split</h3>"
    "<table>"
    "<tr><th>split</th><th>rows</th></tr>"
    f"<tr><td>train</td><td>{len(train_rows)}</td></tr>"
    f"<tr><td>valid</td><td>{len(valid_rows)}</td></tr>"
    f"<tr><td>test</td><td>{len(test_rows)}</td></tr>"
    "</table>"
)
ML_TASK_STATUS = "baseline_ok"


rows: 1000
train_rows: 720
valid_rows: 140
test_rows: 140
positive_rate: 0.4790
profile_artifact: artifacts/agent_rounds/20260615_round6/ml_sweep_realistic/dataset_profile.json
model_artifact: artifacts/agent_rounds/20260615_round6/ml_sweep_realistic/baseline_model.json
metrics_artifact: artifacts/agent_rounds/20260615_round6/ml_sweep_realistic/baseline_metrics.json
baseline_valid:
- tp: 47
- fp: 21
- tn: 54
- fn: 18
- precision: 0.6912
- recall: 0.7231
- accuracy: 0.7214
- f1: 0.7068
- brier: 0.1651
baseline_test:
- tp: 60
- fp: 16
- tn: 50
- fn: 14
- precision: 0.7895
- recall: 0.8108
- accuracy: 0.7857
- f1: 0.8000
- brier: 0.1542


tp,47
fp,21
tn,54
fn,18
precision,0.6912
recall,0.7231
accuracy,0.7214
f1,0.7068
brier,0.1651


split,rows
train,720
valid,140
test,140


In [2]:
from __future__ import annotations

import itertools
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from samples.ml_tasks.common import (  # noqa: E402
    classification_metrics,
    display_html,
    save_json,
    score_rows,
    train_logistic,
    write_csv,
)

ARTIFACT_DIR = Path("artifacts/agent_rounds/20260615_round6/ml_sweep_realistic")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

attempt_path = ARTIFACT_DIR / "sweep_attempt_config.json"
if attempt_path.exists():
    attempt_path.unlink()

learning_rates = [0.035, 0.05, 0.08]
l2_values = [0.0, 0.001, 0.01]
epochs_values = [90, 140]
results = []
models = {}

for lr, l2, epochs in itertools.product(learning_rates, l2_values, epochs_values):
    model = train_logistic(train_rows, lr=lr, l2=l2, epochs=epochs)
    valid_metrics = classification_metrics(score_rows(valid_rows, model))
    result = {"lr": lr, "l2": l2, "epochs": epochs, **valid_metrics}
    results.append(result)
    models[(lr, l2, epochs)] = model
    print(
        f"lr={lr:.3f} l2={l2:.3f} epochs={epochs} "
        f"valid_f1={valid_metrics['f1']:.4f} "
        f"valid_accuracy={valid_metrics['accuracy']:.4f} "
        f"valid_brier={valid_metrics['brier']:.4f}"
    )

best = max(results, key=lambda row: (row["f1"], row["accuracy"], -row["brier"]))
best_model = models[(best["lr"], best["l2"], best["epochs"])]
best_test = classification_metrics(score_rows(test_rows, best_model))

sweep_path = save_json(
    ARTIFACT_DIR / "sweep_results.json",
    {"results": results, "best_validation": best, "best_test": best_test},
)
csv_path = write_csv(
    ARTIFACT_DIR / "sweep_results.csv",
    results,
    ["lr", "l2", "epochs", "tp", "fp", "tn", "fn", "precision", "recall", "accuracy", "f1", "brier"],
)
comparison_path = save_json(
    ARTIFACT_DIR / "comparison_summary.json",
    {
        "baseline_valid": baseline_valid,
        "baseline_test": baseline_test,
        "best_validation": best,
        "best_test": best_test,
        "validation_delta": {
            "f1": best["f1"] - baseline_valid["f1"],
            "accuracy": best["accuracy"] - baseline_valid["accuracy"],
            "brier": best["brier"] - baseline_valid["brier"],
        },
        "test_delta": {
            "f1": best_test["f1"] - baseline_test["f1"],
            "accuracy": best_test["accuracy"] - baseline_test["accuracy"],
            "brier": best_test["brier"] - baseline_test["brier"],
        },
    },
)

print(f"sweep_artifact: {sweep_path}")
print(f"sweep_csv_artifact: {csv_path}")
print(f"comparison_artifact: {comparison_path}")
print("best_validation:")
for key in ["lr", "l2", "epochs", "f1", "accuracy", "precision", "recall", "brier"]:
    value = best[key]
    print(f"- {key}: {value:.4f}" if isinstance(value, float) else f"- {key}: {value}")
print("best_test:")
for key, value in best_test.items():
    print(f"- {key}: {value:.4f}" if isinstance(value, float) else f"- {key}: {value}")
print("validation_delta_vs_baseline:")
print(f"- f1: {best['f1'] - baseline_valid['f1']:.4f}")
print(f"- accuracy: {best['accuracy'] - baseline_valid['accuracy']:.4f}")
print(f"- brier: {best['brier'] - baseline_valid['brier']:.4f}")

rows_html = "\n".join(
    "<tr>"
    f"<td>{row['lr']}</td><td>{row['l2']}</td><td>{row['epochs']}</td>"
    f"<td>{row['f1']:.3f}</td><td>{row['accuracy']:.3f}</td>"
    f"<td>{row['brier']:.3f}</td>"
    "</tr>"
    for row in sorted(results, key=lambda item: item["f1"], reverse=True)
)
display_html(
    "<h3>Validation Sweep Ranking</h3>"
    "<table>"
    "<tr><th>lr</th><th>l2</th><th>epochs</th><th>f1</th><th>accuracy</th><th>brier</th></tr>"
    f"{rows_html}"
    "</table>"
)
ML_TASK_STATUS = "sweep_ok"


lr=0.035 l2=0.000 epochs=90 valid_f1=0.7015 valid_accuracy=0.7143 valid_brier=0.1768
lr=0.035 l2=0.000 epochs=140 valid_f1=0.7015 valid_accuracy=0.7143 valid_brier=0.1684
lr=0.035 l2=0.001 epochs=90 valid_f1=0.7015 valid_accuracy=0.7143 valid_brier=0.1768
lr=0.035 l2=0.001 epochs=140 valid_f1=0.7015 valid_accuracy=0.7143 valid_brier=0.1685
lr=0.035 l2=0.010 epochs=90 valid_f1=0.7015 valid_accuracy=0.7143 valid_brier=0.1772
lr=0.035 l2=0.010 epochs=140 valid_f1=0.7015 valid_accuracy=0.7143 valid_brier=0.1689
lr=0.050 l2=0.000 epochs=90 valid_f1=0.7015 valid_accuracy=0.7143 valid_brier=0.1697
lr=0.050 l2=0.000 epochs=140 valid_f1=0.7068 valid_accuracy=0.7214 valid_brier=0.1650
lr=0.050 l2=0.001 epochs=90 valid_f1=0.7015 valid_accuracy=0.7143 valid_brier=0.1697
lr=0.050 l2=0.001 epochs=140 valid_f1=0.7068 valid_accuracy=0.7214 valid_brier=0.1651
lr=0.050 l2=0.010 epochs=90 valid_f1=0.7015 valid_accuracy=0.7143 valid_brier=0.1702
lr=0.050 l2=0.010 epochs=140 valid_f1=0.7068 valid_accuracy=

lr,l2,epochs,f1,accuracy,brier
0.05,0.0,140,0.707,0.721,0.165
0.05,0.001,140,0.707,0.721,0.165
0.05,0.01,140,0.707,0.721,0.165
0.08,0.0,90,0.707,0.721,0.165
0.08,0.0,140,0.707,0.721,0.164
0.08,0.001,90,0.707,0.721,0.165
0.08,0.001,140,0.707,0.721,0.164
0.08,0.01,90,0.707,0.721,0.165
0.08,0.01,140,0.707,0.721,0.164
0.035,0.0,90,0.701,0.714,0.177


In [3]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from samples.ml_tasks.common import (  # noqa: E402
    classification_metrics,
    display_html,
    save_json,
    score_rows,
)

ARTIFACT_DIR = Path("artifacts/agent_rounds/20260615_round6/ml_sweep_realistic")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

threshold_results = []
for step in range(30, 76, 5):
    threshold = step / 100
    threshold_results.append(
        {"threshold": threshold, **classification_metrics(score_rows(test_rows, best_model, threshold=threshold))}
    )

best_threshold = max(threshold_results, key=lambda row: (row["f1"], row["recall"], row["precision"]))
scored_best = score_rows(test_rows, best_model, threshold=float(best_threshold["threshold"]))
false_positives = sorted(
    [row for row in scored_best if row["label"] == 0 and row["pred"] == 1],
    key=lambda row: row["prob"],
    reverse=True,
)[:6]
false_negatives = sorted(
    [row for row in scored_best if row["label"] == 1 and row["pred"] == 0],
    key=lambda row: row["prob"],
)[:6]

analysis_path = save_json(
    ARTIFACT_DIR / "evaluation_error_analysis.json",
    {
        "threshold_results": threshold_results,
        "best_threshold": best_threshold,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
    },
)

print(f"analysis_artifact: {analysis_path}")
print("best_threshold:")
for key, value in best_threshold.items():
    print(f"- {key}: {value:.4f}" if isinstance(value, float) else f"- {key}: {value}")
print(f"false_positives_reviewed: {len(false_positives)}")
print(f"false_negatives_reviewed: {len(false_negatives)}")
if false_positives:
    print(
        "top_false_positive: "
        f"id={false_positives[0]['id']} prob={false_positives[0]['prob']:.4f} "
        f"support_load={false_positives[0]['support_load']:.4f}"
    )
if false_negatives:
    print(
        "top_false_negative: "
        f"id={false_negatives[0]['id']} prob={false_negatives[0]['prob']:.4f} "
        f"usage={false_negatives[0]['usage']:.4f}"
    )

threshold_rows = "\n".join(
    "<tr>"
    f"<td>{row['threshold']:.2f}</td><td>{row['precision']:.3f}</td>"
    f"<td>{row['recall']:.3f}</td><td>{row['f1']:.3f}</td>"
    f"<td>{row['fp']}</td><td>{row['fn']}</td>"
    "</tr>"
    for row in threshold_results
)
display_html(
    "<h3>Threshold Error Analysis</h3>"
    "<table>"
    "<tr><th>threshold</th><th>precision</th><th>recall</th><th>f1</th><th>fp</th><th>fn</th></tr>"
    f"{threshold_rows}"
    "</table>"
)
ML_TASK_STATUS = "analysis_ok"


analysis_artifact: artifacts/agent_rounds/20260615_round6/ml_sweep_realistic/evaluation_error_analysis.json
best_threshold:
- threshold: 0.4000
- tp: 67
- fp: 24
- tn: 42
- fn: 7
- precision: 0.7363
- recall: 0.9054
- accuracy: 0.7786
- f1: 0.8121
- brier: 0.1490
false_positives_reviewed: 6
false_negatives_reviewed: 6
top_false_positive: id=R00952 prob=0.8469 support_load=-2.1731
top_false_negative: id=R00079 prob=0.2553 usage=0.4348


threshold,precision,recall,f1,fp,fn
0.30,0.682,0.986,0.807,34,1
0.35,0.703,0.959,0.811,30,3
0.40,0.736,0.905,0.812,24,7
0.45,0.768,0.851,0.808,19,11
0.50,0.800,0.811,0.805,15,14
0.55,0.797,0.743,0.769,14,19
0.60,0.831,0.662,0.737,10,25
0.65,0.885,0.622,0.730,6,28
0.70,0.894,0.568,0.694,5,32
0.75,0.912,0.419,0.574,3,43


# Finding

The best validation configuration was `lr=0.08`, `l2=0.01`, `epochs=140`. It matched the baseline validation F1 at `0.7068` and accuracy at `0.7214`, but improved validation Brier score by `0.0009`.

On the held-out test split, the selected sweep model improved F1 from `0.8000` to `0.8054`, accuracy from `0.7857` to `0.7929`, and Brier score from `0.1542` to `0.1490`.

Threshold review found that lowering the decision threshold to `0.40` increased test F1 to `0.8121` with recall `0.9054`, at the cost of more false positives (`24`) than the default threshold. The notebook should treat `0.40` as a candidate operating threshold only if recall is more important than precision.
